# ZS601 clear-glass-free mesh initialization, 200 virtual views

Known Blender clear_glass faces are excluded.1cm cloud:7,004,696 points;3cm training cloud:797,520 points.Original200COLMAP poses,k=3,scale0.5,opacity0.999999,SH0,zero optimization steps.Original coverage masks retained;no semantic image mask.


In [1]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, platform
import torch
ROOT=Path('/content/zs601-mesh-noglass-v006')
PKG=ROOT/'source/gaussian-splatting-lidar-init'
INPUT=ROOT/'input'
RUN=ROOT/'run'
RUN.mkdir(exist_ok=False)
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def save(name,obj):
    with (RUN/name).open('x') as f:json.dump(obj,f,indent=2,allow_nan=False)
def run(command,log):
    print('$',' '.join(map(str,command)),flush=True)
    with (RUN/log).open('x') as f:
        p=subprocess.Popen(list(map(str,command)),cwd=PKG,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:f.write(line);f.flush();print(line,end='',flush=True)
        rc=p.wait()
    if rc:raise RuntimeError(f'{log} failed: {rc}')
env=dict(python=platform.python_version(),torch=torch.__version__,cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0),capability=torch.cuda.get_device_capability(0),
    optimization_steps=0,init_scale_factor=0.5,opacity=0.999999,
    source_commit='4c7186e363f14050c4977bb192f12ed237112764',
    nvidia_smi=subprocess.check_output(['nvidia-smi'],text=True))
save('environment.json',env)
print(json.dumps(env,indent=2))


{
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "cuda": "12.8",
  "gpu": "NVIDIA L4",
  "capability": [
    8,
    9
  ],
  "optimization_steps": 0,
  "init_scale_factor": 0.5,
  "opacity": 0.999999,
  "source_commit": "4c7186e363f14050c4977bb192f12ed237112764",
  "nvidia_smi": "Tue Sep 22 03:32:29 2026       \n+-----------------------------------------------------------------------------------------+\n| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |\n+-----------------------------------------+------------------------+----------------------+\n| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |\n| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |\n|                                         |                        |               MIG M. |\n|=========================================+========================+======================|\n|   0  NVIDIA L4              

In [2]:
source=json.loads((ROOT/'source_manifest.json').read_text())
assert source['code_commit']==env['source_commit']
for r in source['files']:assert sha(ROOT/'source'/r['path'])==r['sha256'],r['path']
inputs=json.loads((INPUT/'input_manifest.json').read_text())
for r in inputs['files']:assert sha(INPUT/r['path'])==r['sha256'],r['path']
assert len(inputs['view_ids'])==200 and inputs['points']==7004696
assert sha(INPUT/'points_mesh_1cm_noglass.ply')=='68f26fca61b4c1424c1f45f2f4983c4c42cf389abc23d6e0afadd9d48819de5b'
save('identity_verified.json',dict(source_commit=source['code_commit'],source_files=len(source['files']),
    input_files=len(inputs['files']),point_cloud_sha256=sha(INPUT/'points_mesh_1cm_noglass.ply')))
print('Immutable renderer and glass-free input verified.')


Immutable renderer and glass-free input verified.


In [3]:
assert sys.version_info[:2]==(3,13) and torch.__version__=='2.11.0+cu128'
assert torch.cuda.get_device_capability(0)==(8,9)
run([sys.executable,'-m','pip','install','plyfile==1.1.3'],'install_python.log')
WHEELS=ROOT/'wheels'
wheels=sorted(WHEELS.glob('*.whl'))
assert len(wheels)==2
run([sys.executable,'-m','pip','install','--no-index','--no-deps']+wheels,'install_cuda.log')
run([sys.executable,'check_contract.py','--sparse',INPUT/'sparse/0'],'check_contract.log')
save('reused_wheels.json',[dict(name=p.name,bytes=p.stat().st_size,sha256=sha(p)) for p in wheels])
save('installed_environment.json',dict(pip_freeze=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)))


$ /usr/bin/python3 -m pip install plyfile==1.1.3


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.8 MB/s eta 0:00:00


$ /usr/bin/python3 -m pip install --no-index --no-deps /content/zs601-mesh-noglass-v006/wheels/diff_gaussian_rasterization-0.0.0-cp313-cp313-linux_x86_64.whl /content/zs601-mesh-noglass-v006/wheels/simple_knn-1.0.0-cp313-cp313-linux_x86_64.whl


Processing /content/zs601-mesh-noglass-v006/wheels/diff_gaussian_rasterization-0.0.0-cp313-cp313-linux_x86_64.whl


Processing /content/zs601-mesh-noglass-v006/wheels/simple_knn-1.0.0-cp313-cp313-linux_x86_64.whl


$ /usr/bin/python3 check_contract.py --sparse /content/zs601-mesh-noglass-v006/input/sparse/0


{'camera_count': 200, 'max_projection_error_px': 2.0463630789890885e-12, 'uint16_mm_roundtrip': True}


In [4]:
preflight=RUN/'preflight'
run([sys.executable,'render_from_sparse_v4.py','--point-cloud',INPUT/'points_mesh_1cm_noglass.ply',
     '--sparse',INPUT/'sparse/0','--output',preflight,'--opacity','0.999999',
     '--init-scale-factor','0.5','--view-ids','3193,3301'],'preflight_render.log')
run([sys.executable,'verify_outputs.py','--point-cloud',INPUT/'points_mesh_1cm_noglass.ply',
     '--sparse',INPUT/'sparse/0','--output',preflight,'--expected-views','2'],'preflight_verify.log')
assert json.loads((preflight/'verification.json').read_text())['views']==2
save('PREFLIGHT_COMPLETE.json',dict(views=2,structural_checks_passed=True,visual_evaluation_pending=True))


$ /usr/bin/python3 render_from_sparse_v4.py --point-cloud /content/zs601-mesh-noglass-v006/input/points_mesh_1cm_noglass.ply --sparse /content/zs601-mesh-noglass-v006/input/sparse/0 --output /content/zs601-mesh-noglass-v006/run/preflight --opacity 0.999999 --init-scale-factor 0.5 --view-ids 3193,3301


Number of points at initialisation :  7004696


{"index": 1, "image_id": 3193, "name": "003193.png", "visible_gaussians": 166813, "coverage_alpha95": 0.9983430166967509, "coverage_alpha50": 1.0, "seconds": 1.583643913269043}


{"index": 2, "image_id": 3301, "name": "003301.png", "visible_gaussians": 400100, "coverage_alpha95": 0.9975518953068592, "coverage_alpha50": 1.0, "seconds": 1.536320686340332}


RENDER_COMPLETE


...

    "depth_coverage": 1.0


  },


  "evaluation": "Blender synthetic GT, fixed near views; no real-image reconstruction or optimized 3DGS claims",


  "ssim": "RGB [0,1], Gaussian 11x11 sigma1.5, C1=0.01^2 C2=0.03^2, fully valid windows"


}


In [5]:
full=RUN/'full'
run([sys.executable,'render_from_sparse_v4.py','--point-cloud',INPUT/'points_mesh_1cm_noglass.ply',
     '--sparse',INPUT/'sparse/0','--output',full,'--opacity','0.999999',
     '--init-scale-factor','0.5'],'full_render.log')
run([sys.executable,'verify_outputs.py','--point-cloud',INPUT/'points_mesh_1cm_noglass.ply',
     '--sparse',INPUT/'sparse/0','--output',full,'--expected-views','200'],'full_verify.log')
report=json.loads((full/'verification.json').read_text())
assert report['views']==200 and report['optimization_steps']==0
print(json.dumps(report,indent=2))
assert sha(preflight/'point_cloud/iteration_0/point_cloud.ply')==sha(full/'point_cloud/iteration_0/point_cloud.ply')


$ /usr/bin/python3 render_from_sparse_v4.py --point-cloud /content/zs601-mesh-noglass-v006/input/points_mesh_1cm_noglass.ply --sparse /content/zs601-mesh-noglass-v006/input/sparse/0 --output /content/zs601-mesh-noglass-v006/run/full --opacity 0.999999 --init-scale-factor 0.5


Number of points at initialisation :  7004696


{"index": 1, "image_id": 3001, "name": "003001.png", "visible_gaussians": 1215131, "coverage_alpha95": 0.9991637522563177, "coverage_alpha50": 0.9993301556859205, "seconds": 1.5784611701965332}


{"index": 11, "image_id": 3031, "name": "003031.png", "visible_gaussians": 60848, "coverage_alpha95": 0.977342339801444, "coverage_alpha50": 1.0, "seconds": 1.5948436260223389}


{"index": 21, "image_id": 3061, "name": "003061.png", "visible_gaussians": 2556840, "coverage_alpha95": 0.9998843637184116, "coverage_alpha50": 1.0, "seconds": 1.5417428016662598}


...

  },


  "evaluation": "Blender synthetic GT, fixed near views; no real-image reconstruction or optimized 3DGS claims",


  "ssim": "RGB [0,1], Gaussian 11x11 sigma1.5, C1=0.01^2 C2=0.03^2, fully valid windows"


}


{
  "status": "VERIFIED_INITIALIZATION_NOT_TRAINED",
  "views": 200,
  "points": 7004696,
  "optimization_steps": 0,
  "exact_xyz": true,
  "max_sh0_rgb_error": 7.807039748009004e-08,
  "opacity_decoded_min": 0.9999989867206773,
  "opacity_decoded_max": 0.9999989867206773,
  "png_count": 1400,
  "camera_pose_and_intrinsics_exact": true,
  "means_over_views": {
    "alpha95_coverage": 0.9974954450586643,
    "depth_coverage": 0.9997759617554152
  },
  "evaluation": "Blender synthetic GT, fixed near views; no real-image reconstruction or optimized 3DGS claims",
  "ssim": "RGB [0,1], Gaussian 11x11 sigma1.5, C1=0.01^2 C2=0.03^2, fully valid windows"
}


# ZS601 clear-glass-free mesh initialization, 200 virtual views

Known Blender clear_glass faces are excluded.1cm cloud:7,004,696 points;3cm training cloud:797,520 points.Original200COLMAP poses,k=3,scale0.5,opacity0.999999,SH0,zero optimization steps.Original coverage masks retained;no semantic image mask.


In [6]:
import numpy as np
from PIL import Image
rows=[]
for folder in ['no_coverage_masks','low_coverage_masks','training_masks_nonempty']:
    (full/folder).mkdir(exist_ok=False)
for path in sorted((full/'alpha').glob('*.png')):
    a=np.asarray(Image.open(path));valid=np.asarray(Image.open(full/'masks'/path.name))==255
    empty=a==0;low=~valid
    for folder,mask in [('no_coverage_masks',empty),('low_coverage_masks',low),('training_masks_nonempty',~empty)]:
        out=full/folder/path.name;Image.fromarray(mask.astype('uint8')*255).save(out)
        assert np.array_equal(np.asarray(Image.open(out))==255,mask)
    rows.append(dict(name=path.name,empty_pixels=int(empty.sum()),empty_fraction=float(empty.mean()),
        low_coverage_fraction=float(low.mean()),alpha_mean=float(a.mean()/65535)))
assert len(rows)==200
save('coverage_masks.json',dict(views=200,per_view=rows,
    empty_definition='alpha16==0; white 255 means no rendered contribution',
    low_definition='float alpha<0.95, inverse of existing white-valid masks',
    training_masks_nonempty='white 255 means keep alpha16>0; black 0 means ignore'))
save('FULL_COMPLETE.json',dict(status='FULL_200_RENDERED_AND_REMOTE_VERIFIED',views=200,
    png_count=2000,optimization_steps=0,init_scale_factor=0.5,formal_200_authorized=True,
    source_commit=env['source_commit'],local_gt_evaluation_pending=True))
print('FULL_200_RENDERED; all mask PNGs verified; local GT evaluation follows download.')


FULL_200_RENDERED; all mask PNGs verified; local GT evaluation follows download.
